In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA02 - Travel Lodging Entertainment (Public Officials)
   
   BUSINESS QUESTION: 
   During the assessment period, did the unit (i) provide or receive gifts, or anything 
   of value to/from OR (ii) pay for the travel, hospitality or entertainment of POs 
   (Public Officials), their family members or close associates?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. DATA SOURCE: Uses COUPA FILES ONLY. (Finance files are excluded for EBA02).
   3. PUBLIC OFFICIAL FILTER: Filters strictly for rows where the `PublicOfficial` 
      column equals 'Y' or 'YES'.
   4. ACCOUNT PARSING: Parses the 'Account' string (format: xxxx-xxxx-xxxxxx) to 
      extract the Cost Center and Category Code. Uses TRY_CAST to INT to safely 
      strip leading zeros.
   5. TARGET CATEGORY CODES: Hardcoded explicitly to (066, 009, 012, 067, 095, 134, 
      140, 180, 181, 793).
   6. CC MAPPING & THE 793 RULE: Maps to AU_IDs using the bootstrap. Applies the 
      exception rule that Category Code 793 is ONLY valid for AU 101016.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Coupa_Raw AS (
    -- 2. Union all 7 Coupa files into one master dataset (No Finance files)
    SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, PublicOfficial FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_PO_Transactions AS (
    -- 3 & 4. Parse the Account string and STRICTLY filter for Public Officials
    SELECT 
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        TRIM(PublicOfficial) AS PublicOfficial
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
      -- Catches 'Y' or 'YES'
      AND UPPER(TRIM(PublicOfficial)) IN ('Y', 'YES')
),

Flagged_PO_Transactions AS (
    -- 5. Filter the Public Official transactions against the Hardcoded Category list
    SELECT 
        Cleaned_CC,
        CatCode_Int
    FROM Coupa_PO_Transactions
    WHERE Cleaned_CC IS NOT NULL
      AND CatCode_Int IN (66, 9, 12, 67, 95, 134, 140, 180, 181, 793)
),

CC_Mapping AS (
    -- Standardize the CC Mapping table
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Flagged_AUs AS (
    -- 6. Map the flagged PO transactions to AU_IDs and APPLY THE 793 RULE
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Flagged_PO_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
    WHERE 
        -- THE 793 EXCEPTION RULE: Exclude code 793 if the AU is NOT 101016
        NOT (f.CatCode_Int = 793 AND m.AU_ID != '101016')
    GROUP BY m.AU_ID
)

-- 7. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA02' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA02 - Travel Lodging Entertainment (POs Only) Traceability
   
   PURPOSE: 
   Isolates the data flow for EBA02 to show exactly how Coupa records are filtered 
   for 'PO', whether their Category Codes trigger a flag, how they map to an AU, 
   and whether they get blocked by the strict "793" exception rule.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data for bridge verification
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Combined_Coupa_Raw AS (
    -- 2. Stack all 7 Coupa target files together
    SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account, DocumentType FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3. Extract the CC, Category Code, and Document Type
    SELECT 
        Account AS Raw_String,
        TRY_CAST(TRIM(SPLIT(Account, '-')[0]) AS INT) AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        TRIM(DocumentType) AS Document_Type
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

CC_Mapping AS (
    -- 4. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        TRY_CAST(TRIM(cost_center_id) AS INT) AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

-- 5. Output the explicit tracing details for every transaction
SELECT 
    t.Raw_String AS Original_Record,
    t.Cleaned_CC AS Parsed_Cost_Center,
    t.CatCode_Int AS Parsed_Category_Code,
    t.Document_Type,
    
    -- Flag if the Category Code exists in our 30-item master list
    CASE WHEN t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892) 
         THEN 'YES' ELSE 'NO' END AS Is_Target_Category,
         
    m.AU_ID AS Mapped_AU_ID,
    mast.Master_AU_Name,
    
    -- Trace exactly how the rules are applied step-by-step
    CASE 
        WHEN UPPER(t.Document_Type) != 'PO' THEN '❌ DROPPED: Not a Purchase Order'
        WHEN t.CatCode_Int NOT IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892) THEN '❌ DROPPED: Not a target category'
        WHEN t.CatCode_Int = 793 AND m.AU_ID != '101016' THEN '❌ BLOCKED: 793 only valid for 101016'
        WHEN t.CatCode_Int = 793 AND m.AU_ID = '101016' THEN '✅ KEPT: Valid 793 mapping'
        ELSE '✅ KEPT: Standard Category mapped'
    END AS Pipeline_Status,
    
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'FAILED BRIDGE' ELSE 'SUCCESS' END AS Master_List_Status
    
FROM Coupa_Parsed t
LEFT JOIN CC_Mapping m 
    ON t.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.Master_Numeric_ID

-- Optional: Uncomment below to view only the successful records
-- WHERE UPPER(t.Document_Type) = 'PO' 
--   AND t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
ORDER BY t.Cleaned_CC;